# Ponto Falante · Visão — treino customizado do YOLO (Google Colab)

**Global Solution 2026.2 · Applied Computer Vision · "Enxergando o mundo real"**

Este notebook treina um detector com duas classes **customizadas**:

| classe | por quê |
|---|---|
| `onibus` | saber que um ônibus está chegando ao ponto |
| `letreiro` | achar a região do letreiro para o OCR ler o número da linha (o YOLO pré-treinado **não** conhece essa classe) |

**Antes de rodar:**
1. `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4` (gratuito).
2. Crie uma conta gratuita no [Roboflow](https://roboflow.com), copie sua *Private API Key* e salve nos **Secrets** do Colab (ícone de chave 🔑 à esquerda) com o nome `ROBOFLOW_API_KEY`. Nunca cole a chave no código.
3. Ajuste a célula **Configuração** (link do seu repositório e as fontes do Roboflow — veja `docs/dataset.md`).
4. `Ambiente de execução > Executar tudo`. O treino leva ~20–40 min na T4.

Tudo que é gerado vai para `resultados/` e `modelos/`, e no fim baixa um `.zip` para você colocar no GitHub.

In [ ]:
!nvidia-smi -L || echo "Sem GPU: troque o ambiente para T4"
!pip install -q ultralytics fiftyone easyocr roboflow reportlab

## 1. Configuração

In [ ]:
# >>> AJUSTE AQUI <<<
REPO_URL = "https://github.com/SEU-USUARIO/visao-ponto-falante.git"   # seu repositório (público)

SEMENTE = 42
MAX_ONIBUS_OPENIMAGES = 400     # por split do Open Images (validation e test) -> até ~800 imagens

# Fontes do Roboflow Universe (formato: workspace, projeto, versão). Confira classes e licença
# no site antes de usar (docs/dataset.md). A PRIMEIRA deve ser o seu projeto com fotos de
# ônibus de São Paulo (Wikimedia Commons) rotuladas por você.
FONTES_ROBOFLOW = [
    {"nome": "sp_proprio", "workspace": "SEU-WORKSPACE", "projeto": "ponto-falante-letreiro", "versao": 1,
     "licenca": "CC BY / CC BY-SA (Wikimedia Commons) — ver docs/creditos_imagens.csv"},
    # Exemplos de datasets públicos com letreiro/número de ônibus (preencha com os que você conferir):
    # {"nome": "sbs_bus_numbers", "workspace": "fulgore", "projeto": "sbs-bus-numbers-uqimt-x9n31", "versao": 1, "licenca": "CC BY 4.0"},
]

# Modelo base: tenta o mais novo e cai para versões anteriores se não existir.
MODELOS_BASE = ["yolo26n.pt", "yolo11n.pt", "yolov8n.pt"]
EPOCAS, TAMANHO_IMG, LOTE = 60, 640, 16

CLASSES = ["onibus", "letreiro"]

In [ ]:
import os, sys, json, shutil, random, hashlib, glob, csv
from pathlib import Path

os.chdir("/content")
if not Path("repo").exists():
    !git clone -q {REPO_URL} repo
os.chdir("/content/repo")
sys.path.insert(0, "/content/repo/src")

BRUTO = Path("/content/dados_brutos"); BRUTO.mkdir(exist_ok=True)
DATASET = Path("/content/dataset")
RES = Path("resultados"); RES.mkdir(exist_ok=True)
Path("modelos").mkdir(exist_ok=True)
random.seed(SEMENTE)
print("Pasta de trabalho:", os.getcwd())

## 2. Fonte A — Open Images V7 (classe *Bus*)
Imagens de ônibus com caixas já anotadas (dataset aberto do Google; anotações CC BY 4.0, imagens listadas como CC BY 2.0).
Baixamos só as imagens de ônibus pelo FiftyOne e convertemos para o formato YOLO com a classe `onibus`.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

def campo_deteccoes(ds):
    for nome, campo in ds.get_field_schema().items():
        if getattr(campo, "document_type", None) is fo.Detections:
            return nome
    raise RuntimeError("Campo de detecções não encontrado")

destino = BRUTO / "openimages"
(destino / "images").mkdir(parents=True, exist_ok=True); (destino / "labels").mkdir(parents=True, exist_ok=True)
total = 0
for split in ["validation", "test"]:
  try:
    ds = foz.load_zoo_dataset("open-images-v7", split=split, label_types=["detections"],
                              classes=["Bus"], max_samples=MAX_ONIBUS_OPENIMAGES, seed=SEMENTE, shuffle=True,
                              dataset_name=f"oi-bus-{split}")
    campo = campo_deteccoes(ds)
    for s in ds:
        dets = [d for d in (s[campo].detections if s[campo] else []) if d.label == "Bus"]
        if not dets:
            continue
        nome = Path(s.filepath).name
        shutil.copy(s.filepath, destino / "images" / nome)
        with open(destino / "labels" / (Path(nome).stem + ".txt"), "w") as f:
            for d in dets:
                x, y, w, h = d.bounding_box          # canto superior esquerdo, relativo
                f.write(f"0 {x + w/2:.6f} {y + h/2:.6f} {w:.6f} {h:.6f}\n")   # 0 = onibus
        total += 1
  except Exception as e:   # não trava o "Executar tudo": segue só com as fontes do Roboflow
    print(f"Open Images ({split}) falhou: {e}")
print("Imagens do Open Images:", total)

## 3. Fonte B — Roboflow (letreiros)
Baixa cada projeto em formato YOLO. A chave vem dos *Secrets* do Colab (nunca do código).

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
baixadas = []
for f in FONTES_ROBOFLOW:
    try:
        proj = rf.workspace(f["workspace"]).project(f["projeto"])
        ds = proj.version(f["versao"]).download("yolov8", location=str(BRUTO / f["nome"]), overwrite=True)
        baixadas.append(f)
        print("OK:", f["nome"])
    except Exception as e:
        print(f"Pulei {f['nome']}: {e}")
print("Fontes Roboflow baixadas:", [f["nome"] for f in baixadas])

## 4. Juntar tudo em um dataset único
Cada fonte usa nomes de classe diferentes. A função `mapear_classe` traduz para as nossas duas classes
(e descarta o resto). Ex.: `BusFront` → `onibus`; `27`, `bus_number`, `sign` → `letreiro`.

Divisão: 70% treino, 20% validação, 10% teste, **feita por fonte** — assim o teste sempre tem fotos de São Paulo.

In [ ]:
import yaml

def mapear_classe(nome: str):
    n = nome.strip().lower()
    if n.isdigit() or any(p in n for p in ["number", "numero", "número", "sign", "letreiro", "board", "route", "destin", "display"]):
        return 1   # letreiro
    if "bus" in n or "onibus" in n or "ônibus" in n:
        return 0   # onibus
    return None    # descarta

def itens_da_fonte(pasta: Path):
    """Devolve [(imagem, [linhas YOLO já remapeadas])]."""
    yaml_path = next(pasta.glob("data.yaml"), None)
    nomes = None
    if yaml_path:
        nomes = yaml.safe_load(open(yaml_path))["names"]
        nomes = list(nomes.values()) if isinstance(nomes, dict) else nomes
    itens = []
    for img in pasta.rglob("*"):
        if img.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"} or "images" not in img.parts:
            continue
        rot = Path(str(img.parent).replace("images", "labels")) / (img.stem + ".txt")
        linhas = []
        if rot.exists():
            for l in rot.read_text().splitlines():
                p = l.split()
                if len(p) != 5:
                    continue   # ignora polígonos/segmentação
                classe = 0 if nomes is None else mapear_classe(str(nomes[int(p[0])]))
                if classe is not None:
                    linhas.append(" ".join([str(classe)] + p[1:]))
        if linhas:
            itens.append((img, linhas))
    return itens

if DATASET.exists():
    shutil.rmtree(DATASET)
composicao, vistos = [], set()
licencas = {"openimages": "Open Images V7 — anotações CC BY 4.0; imagens CC BY 2.0"}
licencas.update({f["nome"]: f.get("licenca", "ver página do projeto") for f in baixadas})

for fonte in sorted(p for p in BRUTO.iterdir() if p.is_dir()):
    itens = []
    for img, linhas in itens_da_fonte(fonte):
        h = hashlib.md5(img.read_bytes()).hexdigest()
        if h not in vistos:            # remove imagens duplicadas entre fontes
            vistos.add(h); itens.append((img, linhas))
    random.shuffle(itens)
    n = len(itens); n_tr, n_va = int(n * 0.7), int(n * 0.2)
    partes = {"train": itens[:n_tr], "val": itens[n_tr:n_tr + n_va], "test": itens[n_tr + n_va:]}
    for split, lista in partes.items():
        (DATASET / split / "images").mkdir(parents=True, exist_ok=True)
        (DATASET / split / "labels").mkdir(parents=True, exist_ok=True)
        for img, linhas in lista:
            nome = f"{fonte.name}__{img.stem}"
            shutil.copy(img, DATASET / split / "images" / (nome + img.suffix.lower()))
            (DATASET / split / "labels" / (nome + ".txt")).write_text("\n".join(linhas) + "\n")
    conta = lambda c: sum(sum(1 for l in ls if l.startswith(f"{c} ")) for _, ls in itens)
    composicao.append({"origem": fonte.name, "imagens": n, "caixas_onibus": conta(0), "caixas_letreiro": conta(1),
                       "train": len(partes["train"]), "val": len(partes["val"]), "test": len(partes["test"]),
                       "licenca": licencas.get(fonte.name, "ver docs/dataset.md")})

data_yaml = {"path": str(DATASET), "train": "train/images", "val": "val/images", "test": "test/images", "names": CLASSES}
yaml.safe_dump(data_yaml, open(DATASET / "data.yaml", "w"), allow_unicode=True)
shutil.copy(DATASET / "data.yaml", RES / "data.yaml")
json.dump(composicao, open(RES / "dataset.json", "w"), ensure_ascii=False, indent=2)
import pandas as pd; pd.DataFrame(composicao)

## 5. Treino customizado
Parâmetros principais (explique no vídeo/relatório):
- **Transfer learning**: começamos de um YOLO pequeno (`n` = nano) pré-treinado no COCO e ajustamos para as nossas 2 classes. Rápido e roda em CPU depois.
- `epochs=60`, `patience=15`: para cedo se a validação parar de melhorar (evita overfitting).
- `hsv_v=0.5`, `hsv_s=0.6`: varia brilho e saturação → robustez a dia/noite/sol forte.
- `degrees=5`, `translate=0.1`, `scale=0.5`: pequenas rotações, deslocamentos e zoom (câmera instalada em ângulos diferentes).
- `fliplr=0.0`: **sem espelhar**, porque letreiro espelhado não existe no mundo real.
- `mosaic=1.0` com `close_mosaic=10`: junta 4 imagens em uma (mais variedade), desligando nas últimas épocas.
- Com `albumentations` instalado, o Ultralytics também aplica desfoque, tons de cinza e CLAHE com baixa probabilidade.

In [ ]:
from ultralytics import YOLO

modelo, base_usada = None, None
for nome in MODELOS_BASE:
    try:
        modelo, base_usada = YOLO(nome), nome
        break
    except Exception as e:
        print(f"{nome} indisponível: {e}")
print("Modelo base:", base_usada)

treino = modelo.train(data=str(DATASET / "data.yaml"), epochs=EPOCAS, imgsz=TAMANHO_IMG, batch=LOTE,
                      patience=15, seed=SEMENTE, project="/content/runs", name="ponto_falante", exist_ok=True,
                      hsv_v=0.5, hsv_s=0.6, degrees=5, translate=0.1, scale=0.5, fliplr=0.0,
                      mosaic=1.0, close_mosaic=10)
PASTA_TREINO = Path(treino.save_dir) if treino else Path("/content/runs/ponto_falante")
shutil.copy(PASTA_TREINO / "weights" / "best.pt", "modelos/best.pt")
for f in ["results.png", "labels.jpg"]:
    if (PASTA_TREINO / f).exists():
        shutil.copy(PASTA_TREINO / f, RES / f)
print("Pesos salvos em modelos/best.pt")

## 6. Avaliação no conjunto de TESTE (imagens que o modelo nunca viu)
- **Precisão**: das caixas que o modelo desenhou, quantas estavam certas.
- **Recall**: dos objetos que existiam, quantos o modelo achou.
- **mAP50**: qualidade geral das caixas (acerto se a sobreposição com o gabarito ≥ 50%).
- **mAP50-95**: o mesmo, exigindo caixas cada vez mais justas (mais rigoroso).

In [ ]:
melhor = YOLO("modelos/best.pt")
m = melhor.val(data=str(DATASET / "data.yaml"), split="test", imgsz=TAMANHO_IMG, project="/content/runs", name="teste", exist_ok=True)

por_classe = {}
for i, idx in enumerate(m.box.ap_class_index):
    p, r, ap50, ap = m.box.class_result(i)
    por_classe[CLASSES[int(idx)]] = {"precisao": round(float(p), 3), "recall": round(float(r), 3),
                                     "map50": round(float(ap50), 3), "map50_95": round(float(ap), 3)}
metricas = {"modelo_base": base_usada, "epocas": EPOCAS, "tamanho_img": TAMANHO_IMG,
            "map50": round(float(m.box.map50), 3), "map50_95": round(float(m.box.map), 3),
            "precisao": round(float(m.box.mp), 3), "recall": round(float(m.box.mr), 3),
            "por_classe": por_classe, "dataset": composicao,
            "velocidade_ms": {k: round(float(v), 1) for k, v in m.speed.items()}}
for f in ["confusion_matrix.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "PR_curve.png"]:
    if (Path(m.save_dir) / f).exists():
        shutil.copy(Path(m.save_dir) / f, RES / f)
json.dump(metricas, open(RES / "metricas.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps(metricas, ensure_ascii=False, indent=2))

## 7. Avaliação do OCR (leitura do número da linha)
1. A célula abaixo cria `resultados/gabarito_ocr.csv` com as imagens de **teste** que têm letreiro.
2. Abra o arquivo (painel de arquivos à esquerda, duplo clique), preencha a coluna `linha_correta` em ~30 linhas
   olhando cada imagem (ex.: `8000-10`). Deixe vazio o que não der para ler nem a olho.
3. Rode a célula seguinte: ela mede acertos, erros e "não consegui ler".

In [ ]:
gab = RES / "gabarito_ocr.csv"
if not gab.exists():
    linhas_csv = []
    for rot in sorted((DATASET / "test" / "labels").glob("*.txt")):
        if any(l.startswith("1 ") for l in rot.read_text().splitlines()):
            img = next((DATASET / "test" / "images").glob(rot.stem + ".*"))
            linhas_csv.append({"imagem": img.name, "linha_correta": ""})
    # prioriza fotos de São Paulo
    linhas_csv.sort(key=lambda d: (not d["imagem"].startswith("sp_"), d["imagem"]))
    with open(gab, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["imagem", "linha_correta"]); w.writeheader(); w.writerows(linhas_csv)
    print(f"Criado {gab} com {len(linhas_csv)} imagens. Preencha 'linha_correta' e rode a próxima célula.")
else:
    print("Gabarito já existe:", gab)

In [ ]:
import cv2
from detector import DetectorOnibus

gabarito = [r for r in csv.DictReader(open(gab)) if r["linha_correta"].strip()]
linhas_ponto = sorted({r["linha_correta"].strip().upper() for r in gabarito})   # simula a lista que viria da API
det = DetectorOnibus("modelos/best.pt", conf=0.25, gpu=True)

acertos = erros = nao_leu = 0
detalhes = []
for r in gabarito:
    frame = cv2.imread(str(DATASET / "test" / "images" / r["imagem"]))
    res = det.detectar(frame, linhas_ponto)
    lida = res.leitura.linha if res.leitura else None
    certo = r["linha_correta"].strip().upper()
    if lida is None: nao_leu += 1; status = "nao_leu"
    elif lida == certo: acertos += 1; status = "acerto"
    else: erros += 1; status = "erro"
    detalhes.append({"imagem": r["imagem"], "correta": certo, "lida": lida, "status": status,
                     "texto_ocr": res.leitura.texto_bruto if res.leitura else ""})

n = len(gabarito)
metricas["ocr"] = {"total": n, "acertos": acertos, "erros": erros, "nao_leu": nao_leu,
                   "taxa_acerto": round(acertos / n, 3) if n else None,
                   "taxa_erro": round(erros / n, 3) if n else None}
json.dump(metricas, open(RES / "metricas.json", "w"), ensure_ascii=False, indent=2)
pd.DataFrame(detalhes).to_csv(RES / "ocr_detalhes.csv", index=False)
print(metricas["ocr"])
pd.DataFrame(detalhes)

## 8. Exemplos de resultados e casos de falha (para o relatório)
Imagens salvas **já anonimizadas** (rostos e placa borrados).

In [ ]:
from privacidade import anonimizar
from demo import desenhar

imgs_teste = sorted((DATASET / "test" / "images").glob("*"))
imgs_teste.sort(key=lambda p: (not p.name.startswith("sp_"), p.name))
for f in RES.glob("exemplo_*.jpg"): f.unlink()
for f in RES.glob("falha_*.jpg"): f.unlink()

exemplos = falhas = 0
for img_path in imgs_teste:
    frame = cv2.imread(str(img_path))
    res = det.detectar(frame, linhas_ponto or None)
    rot = (DATASET / "test" / "labels" / (img_path.stem + ".txt")).read_text()
    tinha_letreiro = any(l.startswith("1 ") for l in rot.splitlines())
    achou_letreiro = any(c == "letreiro" for c, *_ in res.caixas)
    caixas_onibus = [(x1, y1, x2, y2) for c, _, x1, y1, x2, y2 in res.caixas if c == "onibus"]
    vis = desenhar(anonimizar(frame, caixas_onibus), res, "")
    if achou_letreiro and exemplos < 6:
        exemplos += 1; cv2.imwrite(str(RES / f"exemplo_{exemplos}.jpg"), vis)
    elif tinha_letreiro and not achou_letreiro and falhas < 3:
        falhas += 1; cv2.imwrite(str(RES / f"falha_{falhas}.jpg"), vis)
    if exemplos >= 6 and falhas >= 3:
        break
print(f"{exemplos} exemplos e {falhas} falhas salvos em resultados/")

## 9. Gerar o relatório em PDF e baixar tudo
Preencha `relatorio/info.txt` no repositório antes (nome completo e RM de todos).

In [ ]:
!python relatorio/gerar_relatorio.py --resultados resultados --saida resultados/Relatorio_Visao_Ponto_Falante.pdf
!zip -qr /content/entrega_visao.zip resultados modelos/best.pt
from google.colab import files
files.download("/content/entrega_visao.zip")